Section 1:

In [1]:
!nvidia-smi

Fri Oct  9 05:03:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Task1.2

In [6]:
from numba import cuda

device = cuda.get_current_device()
print(f"Device Name: {device.name}")
print(f"Compute Capability: {device.compute_capability}")
print(f"Multiprocessor Count (SMs): {getattr(device, 'MULTIPROCESSOR_COUNT', 'N/A')}")
print(f"Max Threads Per Block: {device.MAX_THREADS_PER_BLOCK}")
print(f"Max Block Dimensions: {device.MAX_BLOCK_DIM_X}, {device.MAX_BLOCK_DIM_Y}, {device.MAX_BLOCK_DIM_Z}")
print(f"Warp Size: {device.WARP_SIZE}")

Device Name: Tesla T4
Compute Capability: ComputeCapability(major=7, minor=5)
Multiprocessor Count (SMs): 40
Max Threads Per Block: 1024
Max Block Dimensions: 1024, 1024, 64
Warp Size: 32


Section 2:

In [7]:
import numpy as np
import time
from numba import cuda


#Task 2.1
# 1. CPU Reference Implementation
def double_cpu_pure(arr):
    out = np.empty_like(arr)
    for i in range(len(arr)):
        out[i] = arr[i] * 2.0
    return out

# 2. GPU Kernel Implementation
@cuda.jit
def double_gpu_kernel(d_in, d_out):
    idx = cuda.grid(1)
    if idx < d_in.shape[0]:
        d_out[idx] = d_in[idx] * 2.0

#Task 2.2
# 3. Benchmark Function
def benchmark_run(N, student_id_seed=1234):
    np.random.seed(student_id_seed)
    h_in = np.random.rand(N).astype(np.float32)

    # Measure CPU Pure Loop (skip for N >= 10M if too slow)
    if N <= 1_000_000:
        t0 = time.perf_counter()
        _ = double_cpu_pure(h_in)
        cpu_time_ms = (time.perf_counter() - t0) * 1000.0
    else:
        cpu_time_ms = float('nan')

    threads_per_block = 256
    blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

    # Warmup compilation run
    d_warmup = cuda.to_device(np.ones(10, dtype=np.float32))
    double_gpu_kernel[1, 32](d_warmup, d_warmup)
    cuda.synchronize()

    # GPU Roundtrip Measurement (Includes PCIe transfers)
    t0 = time.perf_counter()
    d_in = cuda.to_device(h_in)
    d_out = cuda.device_array_like(d_in)
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    h_out = d_out.copy_to_host()
    cuda.synchronize()
    gpu_total_ms = (time.perf_counter() - t0) * 1000.0

    # GPU Compute-Only Measurement (Pre-allocated in VRAM)
    t0 = time.perf_counter()
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    cuda.synchronize()
    gpu_compute_ms = (time.perf_counter() - t0) * 1000.0

    return cpu_time_ms, gpu_total_ms, gpu_compute_ms

# Run benchmarks across required array sizes
sizes = [100, 1000, 10000, 100000, 1000000, 5000000, 10000000]
print(f"{'N':<12} | {'CPU (ms)':<10} | {'GPU Total (ms)':<15} | {'GPU Kernel (ms)':<15} | {'Winner':<8}")
print("-" * 70)

for N in sizes:
    cpu_t, gpu_tot, gpu_k = benchmark_run(N)
    winner = "CPU" if (not np.isnan(cpu_t) and cpu_t < gpu_tot) else "GPU"
    cpu_str = f"{cpu_t:.3f}" if not np.isnan(cpu_t) else "N/A"
    print(f"{N:<12d} | {cpu_str:<10} | {gpu_tot:<15.3f} | {gpu_k:<15.3f} | {winner:<8}")

N            | CPU (ms)   | GPU Total (ms)  | GPU Kernel (ms) | Winner  
----------------------------------------------------------------------


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 40 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


100          | 0.054      | 1.879           | 0.154           | CPU     
1000         | 0.299      | 0.705           | 0.067           | CPU     
10000        | 2.724      | 0.601           | 0.076           | GPU     
100000       | 28.288     | 0.763           | 0.096           | GPU     
1000000      | 254.014    | 4.787           | 0.118           | GPU     
5000000      | N/A        | 13.322          | 0.260           | GPU     
10000000     | N/A        | 25.574          | 0.447           | GPU     


Section 3:

In [10]:
import numpy as np
from numba import cuda

# DEFECTIVE KERNEL: Отсутствует проверка if idx < n!
@cuda.jit
def defective_kernel(d_arr):
    idx = cuda.grid(1)
    # Запись за пределами массива!
    d_arr[idx] = 999.0

N = 1000
h_arr = np.zeros(N, dtype=np.float32)
d_arr = cuda.to_device(h_arr)

threads_per_block = 256
blocks_per_grid = 4 # Launches 4 * 256 = 1024 threads for a 1000-element array

print("Launching defective kernel with 1024 threads on 1000-element array...")
try:
    defective_kernel[blocks_per_grid, threads_per_block](d_arr)
    cuda.synchronize()
    h_result = d_arr.copy_to_host()
    print(f"Execution finished without hard crash. Array len: {len(h_result)}. Last element: {h_result[-1]}")
except Exception as e:
    print(f"Caught CUDA Error: {e}")

Launching defective kernel with 1024 threads on 1000-element array...
Execution finished without hard crash. Array len: 1000. Last element: 999.0


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [11]:
@cuda.jit
def inspect_threads_kernel(log_block, log_thread, log_global, log_active, N):
    t_idx = cuda.threadIdx.x
    b_idx = cuda.blockIdx.x
    g_idx = cuda.grid(1)

    if g_idx < log_block.shape[0]:
        log_block[g_idx] = b_idx
        log_thread[g_idx] = t_idx
        log_global[g_idx] = g_idx
        log_active[g_idx] = 1 if g_idx < N else 0

N = 37
threads_per_block = 8
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block # 5 blocks = 40 threads

log_b = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_t = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_g = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_a = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)

inspect_threads_kernel[blocks_per_grid, threads_per_block](log_b, log_t, log_g, log_a, N)
cuda.synchronize()

print(f"{'Global ID':<10} | {'Block ID':<10} | {'Thread ID':<10} | {'Status':<10}")
print("-" * 50)
for i in range(32, 40):
    status = "ACTIVE" if log_a[i] == 1 else "IDLE (GUARDED)"
    print(f"{log_g[i]:<10} | {log_b[i]:<10} | {log_t[i]:<10} | {status:<10}")

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 5 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


Global ID  | Block ID   | Thread ID  | Status    
--------------------------------------------------
32         | 4          | 0          | ACTIVE    
33         | 4          | 1          | ACTIVE    
34         | 4          | 2          | ACTIVE    
35         | 4          | 3          | ACTIVE    
36         | 4          | 4          | ACTIVE    
37         | 4          | 5          | IDLE (GUARDED)
38         | 4          | 6          | IDLE (GUARDED)
39         | 4          | 7          | IDLE (GUARDED)


Section 4:

In [12]:
import numpy as np
import time
from numba import cuda

# BUGGY NAIVE SUMMATION
@cuda.jit
def naive_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        # DATA RACE: Thousands of threads execute Read-Modify-Write at the exact same clock cycle!
        d_total[0] += d_arr[idx]

# CORRECT ATOMIC SUMMATION
@cuda.jit
def atomic_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        # HARDWARE ATOMIC LOCK: Hardware serializes updates to prevent collisions
        cuda.atomic.add(d_total, 0, d_arr[idx])

N = 50000
h_arr = np.ones(N, dtype=np.float32)

threads_per_block = 256
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

# 1. Run Naive Kernel across 5 consecutive trials
print("--- RUNNING NAIVE KERNEL (5 TRIALS) ---")
naive_results = []
for i in range(5):
    d_arr = cuda.to_device(h_arr)
    d_total = cuda.to_device(np.zeros(1, dtype=np.float32))

    t0 = time.perf_counter()
    naive_sum_kernel[blocks_per_grid, threads_per_block](d_arr, d_total)
    cuda.synchronize()
    naive_time = (time.perf_counter() - t0) * 1000.0

    val = d_total.copy_to_host()[0]
    naive_results.append((val, 50000.0 - val, naive_time))
    print(f"Trial {i+1}: Naive Output = {val:.1f} | Lost Updates = {50000.0 - val:.1f}")

# 2. Run Atomic Kernel
print("\n--- RUNNING ATOMIC KERNEL ---")
d_arr = cuda.to_device(h_arr)
d_total = cuda.to_device(np.zeros(1, dtype=np.float32))

t0 = time.perf_counter()
atomic_sum_kernel[blocks_per_grid, threads_per_block](d_arr, d_total)
cuda.synchronize()
atomic_time = (time.perf_counter() - t0) * 1000.0

atomic_val = d_total.copy_to_host()[0]
print(f"Atomic Sum Output = {atomic_val:.1f}")
print(f"Naive Avg Time = {np.mean([r[2] for r in naive_results]):.3f} ms | Atomic Time = {atomic_time:.3f} ms")

--- RUNNING NAIVE KERNEL (5 TRIALS) ---
Trial 1: Naive Output = 2.0 | Lost Updates = 49998.0
Trial 2: Naive Output = 2.0 | Lost Updates = 49998.0
Trial 3: Naive Output = 2.0 | Lost Updates = 49998.0
Trial 4: Naive Output = 2.0 | Lost Updates = 49998.0
Trial 5: Naive Output = 2.0 | Lost Updates = 49998.0

--- RUNNING ATOMIC KERNEL ---
Atomic Sum Output = 50000.0
Naive Avg Time = 11.573 ms | Atomic Time = 52.476 ms


Section 5:

In [14]:
import math
import numpy as np
from numba import cuda

@cuda.jit
def radial_vignette_2d(d_canvas, width, height, center_x, center_y):
    # Retrieve 2D spatial coordinates directly from the grid
    x, y = cuda.grid(2)
    if x < width and y < height:
        dx = float(x - center_x)
        dy = float(y - center_y)
        dist = math.sqrt(dx * dx + dy * dy)
        max_dist = math.sqrt(float(center_x**2 + center_y**2))
        intensity = 1.0 - (dist / max_dist)
        if intensity < 0.0:
            intensity = 0.0
        d_canvas[y, x] = intensity

# 2D EXECUTION CONFIGURATION
WIDTH, HEIGHT = 1024, 1024
canvas = np.zeros((HEIGHT, WIDTH), dtype=np.float32)
d_canvas = cuda.to_device(canvas)

# 2D Blocks (16x16 = 256 threads per block)
threads_2d = (16, 16)
blocks_2d = (
    (WIDTH + threads_2d[0] - 1) // threads_2d[0],
    (HEIGHT + threads_2d[1] - 1) // threads_2d[1]
)

radial_vignette_2d[blocks_2d, threads_2d](d_canvas, WIDTH, HEIGHT, WIDTH // 2, HEIGHT // 2)
cuda.synchronize()

result_canvas = d_canvas.copy_to_host()

print(f"Blocks in X: {blocks_2d[0]} | Blocks in Y: {blocks_2d[1]} | Total Blocks: {blocks_2d[0] * blocks_2d[1]}")
print(f"Center value (512, 512): {result_canvas[512, 512]:.3f}")
print(f"Outer Corner value (0, 0): {result_canvas[0, 0]:.3f}")

Blocks in X: 64 | Blocks in Y: 64 | Total Blocks: 4096
Center value (512, 512): 1.000
Outer Corner value (0, 0): 0.000


Section 6:

In [15]:
import hashlib

def generate_lab_checksum(student_id_str, d_canvas, exp_table_crossover):
    h = hashlib.sha256()
    h.update(student_id_str.strip().encode('utf-8'))
    # Include GPU name
    h.update(cuda.get_current_device().name.encode('utf-8'))
    # Include sample of compute output
    h.update(d_canvas.copy_to_host()[:10, :10].tobytes())
    h.update(str(exp_table_crossover).encode('utf-8'))
    digest = h.hexdigest()[:16].upper()
    print("=" * 60)
    print(f"OFFICIAL VERIFICATION CHECKSUM TOKEN: {digest}")
    print("=" * 60)
    return digest

# Замените "YOUR_STUDENT_ID" на ваш реальный Student ID
STUDENT_ID = "YOUR_STUDENT_ID"
CROSSOVER_N = 10000  # Точка пересечения из Section 2

generate_lab_checksum(STUDENT_ID, d_canvas, CROSSOVER_N)

OFFICIAL VERIFICATION CHECKSUM TOKEN: D663300D7A1433B4


'D663300D7A1433B4'